# Building an OWL Ontology with owlready2

In this notebook, you will construct a grocery/cookie product ontology using Python and the [owlready2](https://owlready2.readthedocs.io/) library. Each section mirrors a step from the Protege-based class exercise, translating GUI-based ontology construction into programmatic Python code.

**How to use this notebook:** Run each cell in order. The ontology builds progressively — each section depends on the previous ones. Checkpoint cells at the end of each section verify your ontology is in the correct state before moving on.

## Section 1: Setup and Creating an Ontology

An **ontology** is a formal representation of knowledge — it defines the concepts (classes), relationships (properties), and rules (axioms) within a domain. **OWL** (Web Ontology Language) is the W3C standard for expressing ontologies.

In Protege, you create a new ontology via *File > New Ontology* and set its IRI. In owlready2, we do the same thing with `get_ontology()` and the `with onto:` context manager.

**owlready2** is a Python library that provides a Pythonic interface to OWL ontologies — classes become Python classes, properties become Python attributes, and reasoning invokes the HermiT reasoner under the hood.

In [ ]:
from owlready2 import *
import os

# Ensure Java is available for the reasoner (Section 9)
# This points to the JVM installed in the BMDSOWL mamba environment
java_home = os.path.join(os.environ.get("CONDA_PREFIX", ""), "lib", "jvm")
if os.path.exists(java_home):
    os.environ["JAVA_HOME"] = java_home
    os.environ["PATH"] = os.path.join(java_home, "bin") + ":" + os.environ["PATH"]

onto = get_ontology("http://protege.stanford.edu/ontologies/groceries/")

print(f"Ontology IRI: {onto.base_iri}")
print(f"Ontology loaded: {onto}")
print(f"Java available: {os.path.exists(os.path.join(os.environ.get('JAVA_HOME', ''), 'bin', 'java'))}")

The `with onto:` context manager tells owlready2 that any classes or properties declared inside it belong to this ontology. We'll use it throughout the notebook.

In [ ]:
with onto:
    class GroceryItem(Thing): pass
    class FoodStuff(Thing): pass
    class NutritionalInformation(Thing): pass

print("Top-level classes:", list(onto.classes()))

### Checkpoint 1

In [ ]:
assert onto.base_iri == "http://protege.stanford.edu/ontologies/groceries/"
assert len(list(onto.classes())) == 3
assert onto.GroceryItem is not None
assert onto.FoodStuff is not None
assert onto.NutritionalInformation is not None
print("Checkpoint 1 passed!")

## Section 2: Declaring Classes and Class Hierarchy

In OWL, every class is a subclass of `owl:Thing`. You build a taxonomy by declaring **SubClassOf** relationships. In Protege, you use the "Add Subclass" button in the class hierarchy panel.

In owlready2, SubClassOf maps directly to Python class inheritance. `class Dog(Animal)` in Python means `Dog SubClassOf Animal` in OWL.

In [ ]:
with onto:
    class WalkersShortbreadItem(GroceryItem):
        """A grocery store product from the Walkers Shortbread brand."""
        pass

    class WalkersShortbreadFingersBox(WalkersShortbreadItem): pass
    class WalkersShortbreadRoundsBox(WalkersShortbreadItem): pass

print("WalkersShortbreadItem superclasses:", WalkersShortbreadItem.is_a)
print("WalkersShortbreadFingersBox superclasses:", WalkersShortbreadFingersBox.is_a)
print(f"Is WalkersShortbreadFingersBox a GroceryItem? {issubclass(WalkersShortbreadFingersBox, GroceryItem)}")

### Your Turn

Add `LotusBiscoffCookiesItem` as a subclass of `GroceryItem`.

In [ ]:
with onto:
    # TODO: Declare LotusBiscoffCookiesItem as a subclass of GroceryItem
    pass

### Checkpoint 2

In [ ]:
assert issubclass(WalkersShortbreadFingersBox, GroceryItem)
assert issubclass(WalkersShortbreadRoundsBox, GroceryItem)
assert onto.LotusBiscoffCookiesItem is not None, "LotusBiscoffCookiesItem not found — did you declare it?"
assert issubclass(onto.LotusBiscoffCookiesItem, GroceryItem), "LotusBiscoffCookiesItem should be a subclass of GroceryItem"
print(f"Class count: {len(list(onto.classes()))}")
print("Checkpoint 2 passed!")

## Section 3: Building the Ingredient Taxonomy

The file `data/ingredients.txt` contains ~163 ingredient classes in a tab-indented hierarchy. Rather than declaring each class by hand, we'll parse the file and create classes programmatically.

In Protege, this step uses the Cellfie plugin to import from a spreadsheet. Here, we write a short parser that reads the indentation level to determine parent-child relationships, then uses `types.new_class()` to dynamically create owlready2 classes.

In [ ]:
import types as python_types

with onto:
    stack = [(FoodStuff, -1)]

    with open("data/ingredients.txt") as f:
        for line in f:
            if not line.strip():
                continue
            depth = len(line) - len(line.lstrip('\t'))
            name = line.strip()

            while stack and stack[-1][1] >= depth:
                stack.pop()

            parent = stack[-1][0]
            cls = python_types.new_class(name, (parent,))
            stack.append((cls, depth))

print(f"FoodStuff descendants: {len(list(FoodStuff.descendants()))}")

Let's verify a specific hierarchy path to make sure the parser got parent-child relationships right:

In [ ]:
print("WholeOatFlour ancestors:", [c.name for c in onto.WholeOatFlour.ancestors()])
print()
print("Hierarchy path: WholeOatFlour → OatFlour → Flour → FoodStuff → Thing")
assert issubclass(onto.WholeOatFlour, onto.OatFlour)
assert issubclass(onto.OatFlour, onto.Flour)
assert issubclass(onto.Flour, FoodStuff)

### Checkpoint 3

In [ ]:
foodstuff_descendants = list(FoodStuff.descendants())
assert len(foodstuff_descendants) >= 160, f"Expected ~163 FoodStuff descendants, got {len(foodstuff_descendants)}"
assert issubclass(onto.Butter, onto.ButterProduct)
assert issubclass(onto.ButterProduct, onto.DairyProduct)
assert issubclass(onto.DairyProduct, FoodStuff)
assert issubclass(onto.PeppermintOil, onto.EssentialOil)
assert issubclass(onto.EssentialOil, onto.PlantOil)
print(f"Checkpoint 3 passed! ({len(foodstuff_descendants)} FoodStuff descendants)")

## Section 4: Disjoint Classes

By default, OWL uses the **open-world assumption** — just because we haven't said two classes overlap doesn't mean they can't. A **DisjointClasses** axiom explicitly states that two or more classes have no individuals in common. This is critical for reasoning: without disjoint axioms, a reasoner might classify something as both `SeaSalt` and `EvaporatedSalt` simultaneously.

In Protege, you set this in the "Disjoint With" section of the class description panel. In owlready2, we use `AllDisjoint()`.

Since our ingredient hierarchy from `ingredients.txt` is a proper taxonomy, **siblings (children of the same parent) should always be disjoint** — a `SeaSalt` is never an `EvaporatedSalt`, an `Almond` is never a `Cashew`, etc. Rather than declaring each group by hand, we can iterate the hierarchy and declare disjoint siblings automatically.

In [ ]:
with onto:
    # Every set of sibling classes (children of the same parent) should be disjoint.
    # Since ingredients.txt encodes the full hierarchy, we can do this automatically:
    for cls in FoodStuff.descendants():
        children = list(cls.subclasses())
        if len(children) >= 2:
            AllDisjoint(children)

print(f"Disjoint axiom count: {len(list(onto.disjoint_classes()))}")

### Your Turn

Write code to find all classes disjoint with `onto.SeaSalt` by iterating over `onto.disjoint_classes()`. Each disjoint group has an `.entities` attribute containing its members.

In [ ]:
# TODO: Find all classes disjoint with onto.SeaSalt
# Hint: loop over onto.disjoint_classes(), check if onto.SeaSalt is in d.entities
for d in onto.disjoint_classes():
    if onto.SeaSalt in d.entities:
        pass  # TODO: print the other members of this disjoint group

### Checkpoint 4

In [ ]:
disjoint_count = len(list(onto.disjoint_classes()))
assert disjoint_count >= 25, f"Expected at least 25 disjoint axioms, got {disjoint_count}"
print(f"Checkpoint 4 passed! ({disjoint_count} disjoint axioms)")

## Section 5: Annotations

**Annotations** are metadata attached to ontology entities — they provide human-readable information but don't affect logical reasoning. Common annotation properties include:

- `rdfs:label` — human-readable name (supports multiple languages)
- `skos:altLabel` — alternative name (e.g., "Baking Soda" for SodiumBicarbonate)
- `skos:definition` — textual definition
- `foaf:depiction` — image URL
- `rdfs:seeAlso` — link to an external resource
- `dcterms:creator` — who created the entity

In Protege, these appear in the Annotations tab. In owlready2, we access them as Python attributes after importing the relevant namespaces.

In [ ]:
# Import annotation properties from standard vocabularies
SKOS = get_ontology("http://www.w3.org/2004/02/skos/core#")
FOAF = get_ontology("http://xmlns.com/foaf/0.1/")
DCTERMS = get_ontology("http://purl.org/dc/terms/")
SCHEMA = get_ontology("http://schema.org/")

with onto:
    class altLabel(AnnotationProperty):
        namespace = SKOS
    class definition(AnnotationProperty):
        namespace = SKOS
    class depiction(AnnotationProperty):
        namespace = FOAF
    class creator(AnnotationProperty):
        namespace = DCTERMS
    class gtin12(AnnotationProperty):
        namespace = SCHEMA

Now let's annotate `SodiumBicarbonate` with multilingual labels, an alt label, a definition, an image, and a Wikipedia link. owlready2 uses `locstr()` for language-tagged strings:

In [ ]:
onto.SodiumBicarbonate.label = [
    locstr("Sodium Bicarbonate", lang="en"),
    locstr("Bicarbonato de Sodio", lang="es"),
    locstr("Natriumbicarbonat", lang="de"),
]

onto.SodiumBicarbonate.altLabel = [
    locstr("Baking Soda", lang="en"),
    locstr("Backpulver", lang="de"),
]

onto.SodiumBicarbonate.definition = [
    locstr("Sodium bicarbonate, referred to as 'baking soda', is primarily used in baking, "
           "as a leavening agent.", lang="en")
]

onto.SodiumBicarbonate.seeAlso = ["https://en.wikipedia.org/wiki/Sodium_bicarbonate"]
onto.SodiumBicarbonate.depiction = ["https://imgur.com/rKr2e6v.jpg"]

print("Labels:", onto.SodiumBicarbonate.label)
print("Alt labels:", onto.SodiumBicarbonate.altLabel)

We can also annotate the ontology itself and add product identifiers:

In [ ]:
GroceryItem.creator = ["Your Name Here"]

onto.WalkersShortbreadFingersBox.gtin12 = ["039047001152"]
onto.WalkersShortbreadFingersBox.seeAlso = ["http://www.upcitemdb.com/upc/39047001152"]

print("GroceryItem creator:", GroceryItem.creator)
print("Walkers Fingers GTIN-12:", onto.WalkersShortbreadFingersBox.gtin12)

### Your Turn

Annotate `onto.Cinnamon` with:
- An English label `"Cinnamon"` and a Spanish label `"Canela"`
- A definition of your choice (in English)

See the [owlready2 annotations docs](https://owlready2.readthedocs.io/en/latest/annotations.html) for reference.

In [ ]:
# TODO: Add multilingual labels to onto.Cinnamon
onto.Cinnamon.label = []  # TODO: Replace with a list of locstr() values

# TODO: Add a definition to onto.Cinnamon
onto.Cinnamon.definition = []  # TODO: Replace with a list containing one locstr()

### Checkpoint 5

In [ ]:
assert len(onto.SodiumBicarbonate.label) == 3, "SodiumBicarbonate should have 3 labels (en, es, de)"
assert any("Baking Soda" in str(l) for l in onto.SodiumBicarbonate.altLabel)
assert len(onto.Cinnamon.label) >= 2, "Cinnamon should have at least 2 labels — did you fill in the TODO?"
assert len(onto.Cinnamon.definition) >= 1, "Cinnamon should have a definition — did you fill in the TODO?"
print("Checkpoint 5 passed!")

## Section 6: Object Properties

**Object properties** connect two individuals (or classes, via restrictions). They are the "verbs" of an ontology — relationships like *hasIngredient*, *containsFoodStuff*, or *isMadeIn*.

In Protege, you define these in the Object Properties tab and set their domain, range, and characteristics. In owlready2, you declare them as Python classes inheriting from `ObjectProperty`.

We'll also create a **property hierarchy**: `containsFoodStuff` and `hasIngredient` are both sub-properties of a general `contains` property.

In [ ]:
with onto:
    class contains(ObjectProperty): pass

    class containsFoodStuff(contains):
        domain = [GroceryItem]
        range = [FoodStuff]

    class hasIngredient(contains):
        domain = [FoodStuff]
        range = [FoodStuff]

print("Object properties:", list(onto.object_properties()))
print(f"containsFoodStuff is sub-property of: {containsFoodStuff.is_a}")
print(f"hasIngredient is sub-property of: {hasIngredient.is_a}")

We also need cookie-related classes in our hierarchy. Let's add `Cookie` and `Shortbread` as FoodStuff subclasses, and `WalkersShortbread` as a specific recipe:

In [ ]:
with onto:
    class Cookie(FoodStuff): pass
    class Shortbread(Cookie): pass
    class WalkersShortbread(Shortbread): pass
    WalkersShortbread.comment = [locstr("Shortbread made to the Walkers recipe.", lang="en")]

    WalkersShortbreadItem.is_a.append(containsFoodStuff.some(WalkersShortbread))

print(f"WalkersShortbreadItem restrictions: {WalkersShortbreadItem.is_a}")

### Your Turn

Declare `BiscoffCookie` as a subclass of `Cookie`, then link `LotusBiscoffCookiesItem` to `BiscoffCookie` using the `containsFoodStuff` property (just like we linked `WalkersShortbreadItem` to `WalkersShortbread` above).

See the [owlready2 properties docs](https://owlready2.readthedocs.io/en/latest/properties.html) for reference.

In [ ]:
with onto:
    # TODO: Declare BiscoffCookie as a subclass of Cookie
    pass

    # TODO: Add a containsFoodStuff restriction to LotusBiscoffCookiesItem
    # Hint: onto.LotusBiscoffCookiesItem.is_a.append(containsFoodStuff.some(...))

### Checkpoint 6

In [ ]:
assert onto.BiscoffCookie is not None, "BiscoffCookie not found — did you declare it?"
assert issubclass(onto.BiscoffCookie, Cookie), "BiscoffCookie should be a subclass of Cookie"
biscoff_restrictions = [r for r in onto.LotusBiscoffCookiesItem.is_a if hasattr(r, 'property')]
assert any(r.property == containsFoodStuff for r in biscoff_restrictions), \
    "LotusBiscoffCookiesItem needs a containsFoodStuff restriction"
obj_props = list(onto.object_properties())
assert len(obj_props) >= 3, f"Expected at least 3 object properties, got {len(obj_props)}"
print(f"Checkpoint 6 passed! Object properties: {[p.name for p in obj_props]}")

## Section 7: Modeling Cookie Ingredients

Now we use **existential restrictions** (`someValuesFrom`) to model what ingredients each cookie recipe contains. The OWL axiom `WalkersShortbread SubClassOf hasIngredient some Butter` means "every WalkersShortbread has at least one ingredient that is Butter."

In Protege, this is written in Manchester syntax as `hasIngredient some Butter` in the "SubClass Of" panel. In owlready2, we use `hasIngredient.some(Butter)`.

For products that use "one or more of" a set of oils (common on food labels), we model this with a **union**: `hasIngredient some (SoyBeanOil or SunflowerOil or CanolaOil or PalmOil)`.

In [ ]:
with onto:
    # Walkers Shortbread recipe
    WalkersShortbread.is_a.extend([
        hasIngredient.some(onto.Butter),
        hasIngredient.some(onto.WhiteSugar),
        hasIngredient.some(onto.WhiteFlour),
        hasIngredient.some(onto.Salt),
    ])

    # Biscoff Cookie recipe — includes a union for "contains one or more of" vegetable oils
    onto.BiscoffCookie.is_a.extend([
        hasIngredient.some(onto.WheatFlour),
        hasIngredient.some(onto.BrownSugar),
        hasIngredient.some(onto.SodiumBicarbonate),
        hasIngredient.some(onto.SoyFlour),
        hasIngredient.some(onto.Salt),
        hasIngredient.some(onto.Cinnamon),
        hasIngredient.some(onto.SoyBeanOil | onto.SunflowerOil | onto.CanolaOil | onto.PalmOil),
    ])

print("WalkersShortbread axioms:")
for ax in WalkersShortbread.is_a:
    print(f"  {ax}")
print()
print("BiscoffCookie axioms:")
for ax in onto.BiscoffCookie.is_a:
    print(f"  {ax}")

We still have 4 more products to model, each with 10–20 ingredients. Typing every `hasIngredient.some(...)` line by hand would be tedious — and this is exactly where owlready2 beats Protege. In Protege, you'd click through each restriction one at a time. In Python, we can load structured recipe data from a file and apply restrictions in a loop.

The file `data/recipes.json` contains product and sub-ingredient data extracted from the cookie product sheets. Let's load it and build the remaining products programmatically:

In [ ]:
import json

with open("data/recipes.json") as f:
    recipe_data = json.load(f)

with onto:
    for product in recipe_data["products"]:
        # Create GroceryItem subclass and Cookie subclass dynamically
        item_cls = python_types.new_class(product["item_class"], (GroceryItem,))
        parent_cls = onto[product["cookie_parent"]]
        cookie_cls = python_types.new_class(product["cookie_class"], (parent_cls,))

        if product.get("depiction"):
            item_cls.depiction = [product["depiction"]]
        item_cls.is_a.append(containsFoodStuff.some(cookie_cls))

        # Add ingredient restrictions
        for ing_name in product["ingredients"]:
            ing_cls = onto[ing_name]
            assert ing_cls is not None, f"Ingredient '{ing_name}' not found in ontology"
            cookie_cls.is_a.append(hasIngredient.some(ing_cls))

        # Add union ("one or more of") ingredients
        for alt_list in product.get("alt_ingredients", {}).values():
            classes = [onto[name] for name in alt_list]
            union_expr = classes[0]
            for c in classes[1:]:
                union_expr = union_expr | c
            cookie_cls.is_a.append(hasIngredient.some(union_expr))

    # Sub-ingredients: components that themselves have ingredients
    for component_name, ingredients in recipe_data["sub_ingredients"].items():
        comp_cls = onto[component_name]
        for ing_name in ingredients:
            comp_cls.is_a.append(hasIngredient.some(onto[ing_name]))

print(f"Products created from recipes.json:")
for p in recipe_data["products"]:
    cookie_cls = onto[p["cookie_class"]]
    n = len([r for r in cookie_cls.is_a if hasattr(r, 'property')])
    print(f"  {p['item_class']} → {p['cookie_class']} ({n} ingredient restrictions)")
print(f"\nSub-ingredients modeled: {list(recipe_data['sub_ingredients'].keys())}")

In [ ]:
# Let's verify one product to confirm the automation worked
print("FudgeMintCookie axioms:")
for ax in onto.FudgeMintCookie.is_a:
    print(f"  {ax}")
print()
print("Sub-ingredient example — FudgeCoating restrictions:")
for ax in onto.FudgeCoating.is_a:
    if hasattr(ax, 'property'):
        print(f"  {ax}")

### Your Turn

Model a `SemolinaShortbread` recipe as a subclass of `Shortbread`. Based on the recipe, its ingredients are: Butter, WhiteSugar, WhiteFlour, Semolina, and Salt. Add `hasIngredient.some(...)` restrictions for each.

Refer to `data/Cookie-Products.pdf` for full product details.

In [ ]:
with onto:
    # TODO: Declare SemolinaShortbread as a subclass of Shortbread
    pass

    # TODO: Add hasIngredient restrictions for Butter, WhiteSugar, WhiteFlour, Semolina, and Salt
    # Hint: onto.SemolinaShortbread.is_a.extend([
    #     hasIngredient.some(onto.Butter),
    #     ...
    # ])

### Checkpoint 7

In [ ]:
ws_restrictions = [r for r in WalkersShortbread.is_a if hasattr(r, 'property')]
assert len(ws_restrictions) == 4, f"WalkersShortbread should have 4 ingredient restrictions, got {len(ws_restrictions)}"

assert onto.FudgeMintCookie is not None, "FudgeMintCookie should exist (created from recipes.json)"
fmc_restrictions = [r for r in onto.FudgeMintCookie.is_a if hasattr(r, 'property')]
assert len(fmc_restrictions) >= 10, f"FudgeMintCookie should have 10+ restrictions, got {len(fmc_restrictions)}"

assert onto.SemolinaShortbread is not None, "SemolinaShortbread not found — did you declare it?"
assert issubclass(onto.SemolinaShortbread, Shortbread), "SemolinaShortbread should be a subclass of Shortbread"
sem_restrictions = [r for r in onto.SemolinaShortbread.is_a if hasattr(r, 'property')]
assert len(sem_restrictions) >= 5, f"SemolinaShortbread should have at least 5 ingredient restrictions, got {len(sem_restrictions)}"

print("Checkpoint 7 passed!")

## Section 8: Property Characteristics

OWL properties can have characteristics that enable additional inferences:

- **Transitive**: If A contains B and B contains C, then A contains C
- **Sub-property**: If `hasIngredient` is a sub-property of `contains`, then anything with an ingredient also "contains" it
- **Property chain**: If `isMadeIn ∘ isLocatedIn → isMadeIn`, then a product made in Scotland (which is located in Europe) is also made in Europe

In Protege, these are checkboxes and panels in the property editor. In owlready2, we set them as class attributes or use `PropertyChain`.

In [ ]:
with onto:
    # Make "contains" transitive
    contains.is_a.append(TransitiveProperty)

    # Add "contains" links for soy-derived and wheat-derived ingredients
    onto.SoyLecithin.is_a.append(contains.some(onto.Soy))
    onto.SoyFlour.is_a.append(contains.some(onto.Soy))
    onto.SoyBeanOil.is_a.append(contains.some(onto.Soy))
    onto.WheatFlour.is_a.append(contains.some(onto.Wheat))
    onto.Wheat.is_a.append(contains.some(onto.Gluten))

print(f"Is 'contains' transitive? {TransitiveProperty in contains.is_a}")
print()
print("Reasoning chain: WheatFlour contains Wheat, Wheat contains Gluten")
print("Therefore (by transitivity): WheatFlour contains Gluten")

We'll add the `isMadeIn` and `isLocatedIn` properties now (used with individuals in Section 10), along with a property chain. The chain `isMadeIn ∘ isLocatedIn → isMadeIn` means: if a product is made in location X, and X is located in Y, then the product is also made in Y.

In [ ]:
with onto:
    class Place(Thing): pass

    class isMadeIn(ObjectProperty): pass

    class isLocatedIn(ObjectProperty):
        domain = [Place]
        range = [Place]
        is_a = [TransitiveProperty]

    # Property chain: isMadeIn o isLocatedIn -> isMadeIn
    isMadeIn.is_a.append(PropertyChain([isMadeIn, isLocatedIn]))

    # Set domain for hasIngredient
    hasIngredient.domain = [FoodStuff]

print("Property chain declared: isMadeIn ∘ isLocatedIn → isMadeIn")
print(f"isLocatedIn transitive? {TransitiveProperty in isLocatedIn.is_a}")

### Checkpoint 8

In [ ]:
assert TransitiveProperty in contains.is_a, "contains should be transitive"
assert TransitiveProperty in isLocatedIn.is_a, "isLocatedIn should be transitive"
assert any(isinstance(a, PropertyChain) for a in isMadeIn.is_a), "isMadeIn should have a property chain"
print("Checkpoint 8 passed!")

## Section 9: Defined Classes and Automatic Classification

So far, all our classes have been **primitive** — they have *necessary* conditions (SubClassOf) but not *sufficient* conditions. A **defined class** uses `EquivalentClasses` to state necessary AND sufficient conditions: "anything that is a GroceryItem AND contains Soy IS a GroceryItemThatContainsSoy."

This is the key to automatic classification. When you run a reasoner, it checks every class against defined class conditions and reclassifies automatically.

In Protege, you set this in the "Equivalent To" panel and run the reasoner via *Reasoner > Start Reasoner*. In owlready2, we use `equivalent_to` and `sync_reasoner()`.

**Note:** `sync_reasoner()` requires Java to be installed (it invokes the HermiT reasoner).

In [ ]:
with onto:
    class GroceryItemThatContainsSoy(GroceryItem):
        equivalent_to = [GroceryItem & contains.some(onto.Soy)]

    class GroceryItemThatContainsWheat(GroceryItem):
        equivalent_to = [GroceryItem & contains.some(onto.Wheat)]

    class GroceryItemThatContainsDairyProducts(GroceryItem):
        equivalent_to = [GroceryItem & contains.some(onto.DairyProduct)]

    class GroceryItemThatContainsGluten(GroceryItem):
        equivalent_to = [GroceryItem & contains.some(onto.Gluten)]

    class GroceryItemThatContainsNuts(GroceryItem):
        equivalent_to = [GroceryItem & contains.some(onto.NutProduct)]

print("Defined classes created:")
for cls in [GroceryItemThatContainsSoy, GroceryItemThatContainsWheat,
            GroceryItemThatContainsDairyProducts, GroceryItemThatContainsGluten,
            GroceryItemThatContainsNuts]:
    print(f"  {cls.name}: {cls.equivalent_to}")

Now let's run the reasoner to see which products get automatically classified under these allergen categories:

In [ ]:
try:
    sync_reasoner(infer_property_values=True)
    reasoner_available = True
    print("Reasoner ran successfully!\n")

    for defined_cls in [GroceryItemThatContainsSoy, GroceryItemThatContainsWheat,
                        GroceryItemThatContainsDairyProducts, GroceryItemThatContainsGluten,
                        GroceryItemThatContainsNuts]:
        subclasses = [c.name for c in defined_cls.subclasses()]
        print(f"{defined_cls.name}:")
        for s in subclasses:
            print(f"    {s}")
        print()

except Exception as e:
    reasoner_available = False
    print(f"Reasoner not available (Java may not be installed): {e}")
    print()
    print("Expected classification results:")
    print("  GroceryItemThatContainsSoy: Lotus Biscoff, Fudge Mint, Mallomars, Kashi, Pepperidge Farm")
    print("  GroceryItemThatContainsWheat: Walkers, Lotus Biscoff, Fudge Mint, Mallomars, Pepperidge Farm")
    print("  GroceryItemThatContainsDairyProducts: Walkers, Fudge Mint, Mallomars, Pepperidge Farm")
    print("  GroceryItemThatContainsGluten: all products with wheat (via transitivity)")
    print("  GroceryItemThatContainsNuts: Pepperidge Farm Tahoe")

### Checkpoint 9

In [ ]:
assert len(GroceryItemThatContainsSoy.equivalent_to) > 0, "GroceryItemThatContainsSoy should have an equivalent_to definition"
assert len(GroceryItemThatContainsNuts.equivalent_to) > 0

if reasoner_available:
    assert issubclass(onto.PepperidgeFarmTahoeWhiteChocolateMacadamiaCookiesItem, GroceryItemThatContainsNuts), \
        "Pepperidge Farm Tahoe should be classified under GroceryItemThatContainsNuts"
    print("Checkpoint 9 passed! (with reasoner verification)")
else:
    print("Checkpoint 9 passed! (defined classes created; reasoner skipped)")

## Section 10: Individuals

So far we've only created **classes** (abstract categories). **Individuals** (also called instances or named individuals) represent specific things in the world — a particular store, a specific country.

In Protege, you create individuals in the Individuals tab. In owlready2, you create them by calling a class like a constructor: `my_store = Store("WholeFoodsPaloAlto")`.

In [ ]:
with onto:
    class Store(Thing): pass
    class HardwareStore(Store): pass
    class Country(Place): pass
    class Continent(Place): pass

    WholeFoodsPaloAlto = Store("WholeFoodsPaloAlto")

    # Geographic hierarchy — defined as data, created with a loop
    place_types = {
        "Country": ["Belgium", "Canada", "Scotland"],
        "Continent": ["Europe", "NorthAmerica"],
        "Place": ["GreatBritain", "Aberlour", "Speyside"],
    }
    type_map = {"Country": Country, "Continent": Continent, "Place": Place}

    for place_type, names in place_types.items():
        for name in names:
            type_map[place_type](name)

    # Geographic containment
    located_in = {
        "Aberlour": "Speyside", "Speyside": "Scotland",
        "Scotland": "GreatBritain", "GreatBritain": "Europe",
        "Belgium": "Europe", "Canada": "NorthAmerica",
    }
    for place_name, container_name in located_in.items():
        onto[place_name].isLocatedIn = [onto[container_name]]

    # Products linked to manufacturing locations
    product_origins = {
        "LotusBiscoffCookiesItem": "Belgium",
        "MallomarsCookiesPureChocolateItem": "Canada",
        "WalkersShortbreadItem": "Aberlour",
    }
    for product_name, place_name in product_origins.items():
        onto[product_name].is_a.append(isMadeIn.value(onto[place_name]))

# Keep variables accessible for the student exercise cell
Belgium = onto.Belgium
NorthAmerica = onto.NorthAmerica

print("Individuals:", list(onto.individuals()))
print()
print(f"Aberlour isLocatedIn: {onto.Aberlour.isLocatedIn}")
print(f"Scotland isLocatedIn: {onto.Scotland.isLocatedIn}")

### Your Turn

Create a new `Store` individual called `"TraderJoesPaloAlto"` and a new `Country` individual called `"UnitedStates"`. Make `UnitedStates` located in `NorthAmerica`.

In [ ]:
with onto:
    # TODO: Create a Store individual named "TraderJoesPaloAlto"
    # Hint: TraderJoesPaloAlto = Store("TraderJoesPaloAlto")
    pass

    # TODO: Create a Country individual named "UnitedStates"
    pass

    # TODO: Set UnitedStates.isLocatedIn = [NorthAmerica]
    pass

### Checkpoint 10

In [ ]:
individuals = list(onto.individuals())
assert len(individuals) >= 8, f"Expected at least 8 individuals, got {len(individuals)}"
assert onto.WholeFoodsPaloAlto is not None
assert onto.Belgium is not None
assert onto.TraderJoesPaloAlto is not None, "TraderJoesPaloAlto not found — did you create it?"
assert onto.UnitedStates is not None, "UnitedStates not found — did you create it?"
print(f"Checkpoint 10 passed! ({len(individuals)} individuals)")

## Section 11: Data Properties and Nutritional Information

**Data properties** connect individuals to literal values (numbers, strings) rather than to other individuals. We'll use them to record nutritional facts: total fat, saturated fat, sodium, and sugar content per serving.

In Protege, data properties appear in the Data Properties tab. In owlready2, they inherit from `DataProperty` and we assign values directly to individuals.

In [ ]:
with onto:
    class hasTotalFat(DataProperty, FunctionalProperty):
        domain = [NutritionalInformation]
        range = [float]

    class hasSaturatedFat(DataProperty, FunctionalProperty):
        domain = [NutritionalInformation]
        range = [float]

    class hasSodium(DataProperty, FunctionalProperty):
        domain = [NutritionalInformation]
        range = [float]

    class hasSugar(DataProperty, FunctionalProperty):
        domain = [NutritionalInformation]
        range = [float]

    class hasNutritionalInfo(ObjectProperty, FunctionalProperty): pass

print("Data properties:", list(onto.data_properties()))

Now let's create nutritional information individuals and link them to products. We'll do the first one manually to see the pattern, then automate the rest from `data/nutrition.json`:

In [ ]:
with onto:
    # First, one manual example to see the pattern
    walkers_nutr = NutritionalInformation("WalkersShortbreadFingersBoxNutritionalInformation")
    walkers_nutr.hasTotalFat = 6.0
    walkers_nutr.hasSaturatedFat = 4.0
    walkers_nutr.hasSodium = 50.0
    walkers_nutr.hasSugar = 3.0
    onto.WalkersShortbreadFingersBox.is_a.append(hasNutritionalInfo.value(walkers_nutr))

print(f"Walkers Fingers — Fat: {walkers_nutr.hasTotalFat}g, Sodium: {walkers_nutr.hasSodium}mg, Sugar: {walkers_nutr.hasSugar}g")
print()

# Now automate the rest from nutrition.json
with open("data/nutrition.json") as f:
    nutrition_data = json.load(f)

print("Automating nutritional info from nutrition.json...")
with onto:
    for product_name, values in nutrition_data.items():
        if product_name == "WalkersShortbreadFingersBox":
            continue  # Already did this one manually

        nutr = NutritionalInformation(f"{product_name}NutritionalInformation")
        nutr.hasTotalFat = values["totalFat"]
        nutr.hasSaturatedFat = values["saturatedFat"]
        nutr.hasSodium = values["sodium"]
        nutr.hasSugar = values["sugar"]
        onto[product_name].is_a.append(hasNutritionalInfo.value(nutr))
        print(f"  {product_name} — Fat: {values['totalFat']}g, Sodium: {values['sodium']}mg, Sugar: {values['sugar']}g")

# Keep reference for checkpoint
biscoff_nutr = onto["LotusBiscoffCookiesItemNutritionalInformation"]

### Your Turn

The automation above loaded data for the products listed in `nutrition.json`, but `WalkersShortbreadRoundsBox` is not in that file. Manually add its nutritional information — it's the same recipe as the Fingers box, so use the same values (Fat: 5.0g, Saturated Fat: 3.5g, Sodium: 45.0mg, Sugar: 3.0g).

In [ ]:
with onto:
    # TODO: Create a NutritionalInformation individual for WalkersShortbreadRoundsBox
    # Hint:
    # rounds_nutr = NutritionalInformation("WalkersShortbreadRoundsBoxNutritionalInformation")
    # rounds_nutr.hasTotalFat = ...
    # rounds_nutr.hasSaturatedFat = ...
    # rounds_nutr.hasSodium = ...
    # rounds_nutr.hasSugar = ...
    # onto.WalkersShortbreadRoundsBox.is_a.append(hasNutritionalInfo.value(rounds_nutr))
    pass

### Checkpoint 11

In [ ]:
data_props = list(onto.data_properties())
assert len(data_props) == 4, f"Expected 4 data properties, got {len(data_props)}"
assert walkers_nutr.hasTotalFat == 6.0
assert walkers_nutr.hasSodium == 50.0
assert biscoff_nutr.hasSugar == 12.0
print(f"Checkpoint 11 passed! Data properties: {[p.name for p in data_props]}")

## Section 12: Querying and Exporting

Now that our ontology is built, we can query it programmatically and save it to a file that Protege can open.

owlready2 provides `onto.search()` for simple queries and `default_world.sparql_query()` for SPARQL queries (requires `rdflib`). In Protege, these correspond to the DL Query tab and the SPARQL Query tab.

In [ ]:
# Search by label
results = onto.search(label="*Soda*")
print("Search for '*Soda*' in labels:", results)
print()

# Search by class membership
cookies = onto.search(subclass_of=Cookie)
print("All Cookie subclasses:")
for c in cookies:
    print(f"  {c.name}")
print()

# Search for GroceryItem subclasses
grocery_items = onto.search(subclass_of=GroceryItem)
print(f"GroceryItem subclasses: {len(grocery_items)} found")

We can also run SPARQL queries against the ontology. Here's a query to find all grocery items and what food stuff they contain:

In [ ]:
sparql_results = list(default_world.sparql_query("""
    PREFIX grocery: <http://protege.stanford.edu/ontologies/groceries/>
    SELECT ?item ?food WHERE {
        ?item rdfs:subClassOf ?restriction .
        ?restriction owl:onProperty grocery:containsFoodStuff .
        ?restriction owl:someValuesFrom ?food .
    }
"""))

print("Grocery items and what they contain:")
for row in sparql_results:
    print(f"  {row[0].name} → {row[1].name}")

### Your Turn

Write a SPARQL query to find all cookie recipes that have `hasIngredient` restrictions involving any `DairyProduct` subclass. Hint: use `rdfs:subClassOf*` to match subclasses transitively.

In [ ]:
# TODO: Write a SPARQL query to find cookies with dairy ingredients
# Hint: 
# dairy_results = list(default_world.sparql_query("""
#     PREFIX grocery: <http://protege.stanford.edu/ontologies/groceries/>
#     SELECT ?cookie ?ingredient WHERE {
#         ?cookie rdfs:subClassOf grocery:Cookie .
#         ?cookie rdfs:subClassOf ?restriction .
#         ?restriction owl:onProperty grocery:hasIngredient .
#         ?restriction owl:someValuesFrom ?ingredient .
#         ?ingredient rdfs:subClassOf* grocery:DairyProduct .
#     }
# """))
# for row in dairy_results:
#     print(f"  {row[0].name} uses {row[1].name}")
pass

Finally, let's save the ontology to a file. The saved file can be opened in Protege for visual inspection:

In [ ]:
import os
os.makedirs("ontologies", exist_ok=True)

onto.save(file="ontologies/my_grocery_ontology.owl", format="rdfxml")
print("Ontology saved to ontologies/my_grocery_ontology.owl")
print(f"File size: {os.path.getsize('ontologies/my_grocery_ontology.owl'):,} bytes")

Let's verify we can reload the saved ontology:

In [ ]:
onto_reloaded = get_ontology("ontologies/my_grocery_ontology.owl").load()
print(f"Reloaded ontology: {onto_reloaded}")
print(f"Classes: {len(list(onto_reloaded.classes()))}")
print(f"Object properties: {len(list(onto_reloaded.object_properties()))}")
print(f"Data properties: {len(list(onto_reloaded.data_properties()))}")
print(f"Individuals: {len(list(onto_reloaded.individuals()))}")

### Checkpoint 12

In [ ]:
assert os.path.exists("ontologies/my_grocery_ontology.owl"), "Saved ontology file not found"
assert len(list(onto_reloaded.classes())) > 150, "Reloaded ontology should have 150+ classes"
print("Checkpoint 12 passed!")
print()
print("Congratulations! You have built a complete grocery ontology with:")
print(f"  {len(list(onto.classes()))} classes")
print(f"  {len(list(onto.object_properties()))} object properties")
print(f"  {len(list(onto.data_properties()))} data properties")
print(f"  {len(list(onto.individuals()))} individuals")
print(f"  {len(list(onto.disjoint_classes()))} disjoint axioms")